In [0]:
%run ./utils/logger

In [0]:
run_id = get_run_id()
print(run_id)


In [0]:
dbutils.widgets.text('catalog','commerce_stage_dev')
dbutils.widgets.text('schema','silver')
dbutils.widgets.text("env", "dev")

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
env = dbutils.widgets.get("env")

In [0]:
returns_df = spark.table(f'commerce_raw_{env}.bronze.returns')

In [0]:
display(returns_df)

In [0]:
from pyspark.sql.functions import col
returns_filter_df = returns_df.filter(col("product_id").isNotNull())

In [0]:
returns_clean_df = returns_filter_df.distinct()

In [0]:
returns_clean_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{schema}.returns_temp1")

In [0]:
%sql
select * from returns_temp1

In [0]:
spark.sql(f"""
CREATE TABLE if not EXISTS {catalog}.{schema}.returns_stage_dedup as
select *  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY return_id
               ORDER BY ingestion_time DESC
           ) AS rn
    FROM returns_temp1
)
WHERE rn = 1
""");


In [0]:
df = spark.sql(f'describe table extended commerce_stage_{env}.{schema}.return_stage')
display(df)

In [0]:
spark.sql(f"""
MERGE INTO commerce_stage_{env}.{schema}.return_stage rs
USING commerce_stage_{env}.{schema}.returns_stage_dedup rd

ON rs.return_id = rd.return_id

WHEN MATCHED THEN
UPDATE SET
    rs.order_id = rd.order_id,
    rs.product_id = rd.product_id,
    rs.return_date = rd.return_date,
    rs.refund_amount = rd.refund_amount,
    rs.updated_ts = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    return_id,
    order_id,
    product_id,
    return_date,
    return_reason,
    refund_amount,
    created_ts,
    updated_ts
)
VALUES (
    rd.return_id,
    rd.order_id,
    rd.product_id,
    rd.return_date,
    NULL,
    rd.refund_amount,
    current_timestamp(),
    current_timestamp()
)
""")

In [0]:
spark.sql(f"""
          drop table if exists commerce_stage_{env}.{schema}.returns_temp1
          """)

In [0]:
spark.sql(f"""
          drop table if exists commerce_stage_{env}.{schema}.returns_stage_dedup
          """)